In [1]:
from matplotlib.ticker import LinearLocator, MaxNLocator
import matplotlib.pyplot as plt
from pathlib import Path
import sys

import numpy as np

try:
    import torch
except ModuleNotFoundError:
    torch = None

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

try:
    from sbi.analysis import pairplot
    from sbi.analysis.plotting_classes import KdeOffDiagOptions
    from sbi.analysis.plotting_classes import (
        HistDiagOptions,
        HistOffDiagOptions,
        FigOptions,
        ScatterOffDiagOptions,
        ContourOffDiagOptions,
    )
    from sbi.inference import NPE
except ModuleNotFoundError:
    pairplot = None
    KdeOffDiagOptions = None
    HistDiagOptions = None
    HistOffDiagOptions = None
    FigOptions = None
    NPE = None


In [2]:
prepared_theta_path = "" 
prepared_x_path = ""
prepared_obs_path = "" 
prepared_prior_path = ""
posterior_save_path = r"outputs\posteriors\posterior_16e3emul_N4e4_binned16_s42_xo_Planck_synthetic_no_noise.npy"
plot_summary_path = "training_validation_loss.png"
scale = 1.3813351099973066


In [ ]:
prepared_theta = np.load(prepared_theta_path)
prepared_x = np.load(prepared_x_path)
prepared_obs = np.load(prepared_obs_path)
prepared_prior = np.load(prepared_prior_path)

In [ ]:
seed = 42
torch.manual_seed(seed)

# choose sbi method and train
inference = NPE(prior=prepared_prior, density_estimator="zuko_maf")
density_estimator = inference.append_simulations(
    prepared_theta,
    prepared_x + np.log10(scale),
).train(stop_after_epochs=60)

# do inference given observed data
# x_o = prepared["x_observed_noisy_log10"]
x_o = prepared_obs + np.log10(scale)
posterior = inference.build_posterior(density_estimator)
samples = posterior.sample((100000,), x=x_o)


In [ ]:
np.save(posterior_save_path, samples)

In [ ]:
out = plot_summary(
    inference,
    tags=["training_loss", "validation_loss"],
    figsize=(10, 2),
)

# Robust to whether sbi returns fig or (fig, ax/axes)
fig = out[0] if isinstance(out, tuple) else out

fig.savefig(plot_summary_path, dpi=300, bbox_inches="tight")
plt.close(fig)  # optional: avoids many open figures in loops